# Week 3-4 · orchestration 실패 불변조건 평가

## 시나리오
빈 proposal, 혼합 안전/위험 proposal, review 한도 소진, 잘못된 severity를 표 기반으로 검증해 어떤 실패에서도 명령이 실행되지 않도록 합니다.

## 학습 목표
- 실패 입력과 기대 상태를 case로 선언한다.
- 축소 safety pipeline을 각 case에 적용한다.
- severity 검증과 fail-closed·비실행 불변조건을 검사한다.

## 직접 조립
완성된 `weekX.app` 함수를 가져오지 않습니다. 아래 코드에서 작은 fixture와 핵심 객체·함수·연결을 직접 만듭니다.

### 1단계 · 축소 안전 pipeline

In [ ]:
# 실행 순서: 1단계 · 축소 안전 pipeline에서 practice_validate_severity, practice_incident_guard을(를) 먼저 구성합니다.
# 관찰 포인트: 이 셀의 출력이 다음 단계에서 사용할 입력 계약을 충족하는지 확인합니다 — 1단계 · 축소 안전 pipeline.
SAFE_PREFIXES = ("observe ", "kubectl get ", "kubectl describe ", "kubectl logs ", "kubectl rollout status ")
CONTROL_MARKERS = (";", "&", "|", "`", "$", "<", ">", chr(92), chr(10), chr(13))
ALLOWED_SEVERITIES = {"SEV1", "SEV2", "SEV3"}

# 지원 severity 집합 밖의 값이 safety pipeline에 진입하지 못하게 합니다.
def practice_validate_severity(severity: str) -> str:
    if severity not in ALLOWED_SEVERITIES:
        raise ValueError(f"unsupported severity: {severity}")
    return severity

# proposal batch·revision budget·severity를 함께 평가해 승인·관찰·차단 상태를 만듭니다.
def practice_incident_guard(proposals: list[str], revision_count: int, max_revisions: int, severity: str = "SEV1") -> dict:
    practice_validate_severity(severity)
    safe = [command for command in proposals if command.startswith(SAFE_PREFIXES) and not any(marker in command for marker in CONTROL_MARKERS)]
    all_safe = bool(proposals) and len(safe) == len(proposals)
    if all_safe and severity == "SEV3":
        status = "plan_ready"
    elif all_safe:
        status = "human_approval_required"
    elif revision_count >= max_revisions or not proposals:
        status = "blocked_manual_review"
    else:
        status = "revision_required"
    if status == "human_approval_required":
        approval_packet = {"proposed_commands": safe}
    elif status == "plan_ready":
        approval_packet = None
    else:
        approval_packet = {"proposed_commands": []}
    observation_plan = {"proposed_commands": safe} if status == "plan_ready" else None
    return {
        "status": status,
        "approval_packet": approval_packet,
        "observation_plan": observation_plan,
        "executed_commands": [],
    }

### 2단계 · failure matrix 실행

In [ ]:
# 실행 순서: 2단계 · failure matrix 실행에서 fixture와 assertion을(를) 먼저 구성합니다.
# 관찰 포인트: 이 셀의 출력이 다음 단계에서 사용할 입력 계약을 충족하는지 확인합니다 — 2단계 · failure matrix 실행.
failure_cases = [
    ({"proposals": [], "revision_count": 0, "max_revisions": 1}, "blocked_manual_review"),
    ({"proposals": ["kubectl delete namespace production"], "revision_count": 0, "max_revisions": 1}, "revision_required"),
    ({"proposals": ["kubectl delete namespace production"], "revision_count": 1, "max_revisions": 1}, "blocked_manual_review"),
    ({"proposals": ["kubectl get pods; kubectl delete pod checkout"], "revision_count": 1, "max_revisions": 1}, "blocked_manual_review"),
    ({"proposals": ["kubectl get pods -n shop", "kubectl delete namespace production"], "revision_count": 1, "max_revisions": 1}, "blocked_manual_review"),
]
practice_results = [practice_incident_guard(**inputs) for inputs, _ in failure_cases]
[(result["status"], result["executed_commands"]) for result in practice_results]

### 3단계 · 불변조건 assertions

In [ ]:
# 실행 순서: 3단계 · 불변조건 assertions에서 fixture와 assertion을(를) 먼저 구성합니다.
# 관찰 포인트: 이 셀의 출력이 다음 단계에서 사용할 입력 계약을 충족하는지 확인합니다 — 3단계 · 불변조건 assertions.
for result, (_, expected_status) in zip(practice_results, failure_cases):
    assert result["status"] == expected_status
    assert result["executed_commands"] == []
    assert result["approval_packet"]["proposed_commands"] == []

safe_review = practice_incident_guard(["kubectl get pods -n shop"], revision_count=0, max_revisions=1, severity="SEV2")
assert safe_review["status"] == "human_approval_required"
assert safe_review["executed_commands"] == []
sev3_plan = practice_incident_guard(["observe checkout error-rate dashboard"], revision_count=0, max_revisions=1, severity="SEV3")
assert sev3_plan["status"] == "plan_ready"
assert sev3_plan["approval_packet"] is None
assert sev3_plan["observation_plan"]["proposed_commands"] == ["observe checkout error-rate dashboard"]
assert sev3_plan["executed_commands"] == []
assert practice_validate_severity("SEV3") == "SEV3"
try:
    practice_validate_severity("SEV0")
except ValueError as error:
    practice_severity_error = str(error)
assert "unsupported severity" in practice_severity_error
{"failure_cases": len(failure_cases), "safe_review": safe_review, "sev3_plan": sev3_plan, "invalid_severity": practice_severity_error, "blocked_manual_review": True}

## 중간 결과
각 코드 셀의 출력에서 입력이 어떤 상태로 변했는지 확인합니다. 마지막 `assert`는 눈으로 본 결과를 실행 가능한 계약으로 고정합니다.

## 실패 경계
빈 출력이나 위험 출력은 그럴듯한 성공 packet으로 바꾸지 않습니다. `executed_commands`는 입력과 관계없이 항상 빈 목록입니다.

## 실제 app 연결
Week 3 app의 고급 graph를 그대로 호출하지 않고 최종 안전 계약만 별도 evaluator로 재현합니다. 이 계약은 역할 node나 loop 구현이 바뀌어도 유지돼야 합니다.

### 확장 과제
fixture의 문장이나 임계값을 하나 바꾸고, 어느 중간 결과와 assertion이 달라지는지 기록하세요.

## 다음 Notebook 연결
이 Notebook은 3주 과정의 마지막 실습입니다. 다음 단계에서는 실제 app test와 비교해 축소 실습의 불변조건이 유지되는지 확인합니다.